In [17]:
!pip install imutils


In [21]:
import os

# Шлях до репо
LPRNET_REPO_PATH = 'LPRNet_Pytorch'

# Повертаємося в корінь проекту (про всяк випадок)
while not os.path.exists(LPRNET_REPO_PATH) and os.getcwd() != '/':
    os.chdir('..')

print(f"Поточна папка: {os.getcwd()}")

# Відкат змін git
try:
    os.chdir(LPRNET_REPO_PATH)
    !git checkout .
    print("Репозиторій відновлено до оригінального стану.")
except Exception as e:
    print(f"Помилка відновлення (можливо, це не git репо?): {e}")
finally:
    # Повертаємося назад у корінь проекту
    os.chdir('..')

Поточна папка: /mnt/c/Users/user/PycharmProjects/Diploma/try5
Updated 2 paths from the index
Репозиторій відновлено до оригінального стану.


In [1]:
import sys
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time

LPRNET_REPO_PATH = 'LPRNet_Pytorch'
IMG_SIZE = (94, 24)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
LPR_MAX_LEN = 18

TRAIN_DIR = 'lpr_datasets_v2/lpr_crops/train'
VAL_DIR = 'lpr_datasets_v2/lpr_crops/val'
SAVE_DIR = 'weights_v2'

CHARS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '.']
CHARS_DICT = {char: i for i, char in enumerate(CHARS)}
NUM_CLASS = len(CHARS)

os.makedirs(SAVE_DIR, exist_ok=True)

if LPRNET_REPO_PATH not in sys.path:
    sys.path.append(LPRNET_REPO_PATH)

try:
    from model.LPRNet import LPRNet
    print("Модель LPRNet успішно імпортовано.")
except ImportError:
    print("Помилка: Не можу знайти model/LPRNet.py.")

class CustomLPRDataset(Dataset):
    def __init__(self, img_dir, img_size, chars_dict):
        self.img_dir = img_dir
        self.img_paths = []
        self.labels = []
        self.img_size = img_size
        self.chars_dict = chars_dict

        if not os.path.exists(img_dir):
            print(f"УВАГА: Папка {img_dir} не існує!")
            return

        files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
        print(f"Знайдено файлів у {img_dir}: {len(files)}")

        for filename in files:
            try:
                base = os.path.splitext(filename)[0]
                parts = base.split('_')

                if len(parts) >= 2:
                    if parts[-1].isdigit() and len(parts) > 2:
                         label = parts[-2]
                    else:
                         label = parts[-1]
                else:
                    label = base

                if all(c in chars_dict for c in label):
                    self.img_paths.append(os.path.join(img_dir, filename))
                    self.labels.append(label)
            except Exception:
                continue

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, index):
        path = self.img_paths[index]
        label = self.labels[index]

        image = cv2.imread(path)
        if image is None:
             image = np.zeros((self.img_size[1], self.img_size[0], 3), dtype=np.uint8)

        image = cv2.resize(image, self.img_size)
        image = image.astype('float32')
        image -= 127.5
        image *= 0.0078125
        image = np.transpose(image, (2, 0, 1))

        return torch.from_numpy(image), label

def collate_fn(batch):
    imgs = []
    labels = []
    lengths = []
    for _, (img, label) in enumerate(batch):
        imgs.append(img)
        labels.extend([CHARS_DICT[c] for c in label])
        lengths.append(len(label))
    labels = np.asarray(labels).flatten().astype(np.int32)
    return torch.stack(imgs), torch.from_numpy(labels), lengths

def sparse_tuple_for_ctc(T_length, lengths):
    input_lengths = []
    target_lengths = []
    for l in lengths:
        target_lengths.append(l)
        input_lengths.append(T_length)
    return tuple(input_lengths), tuple(target_lengths)

def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Використовуємо пристрій: {device}")

    print("Завантаження Train...")
    train_ds = CustomLPRDataset(TRAIN_DIR, IMG_SIZE, CHARS_DICT)
    print("Завантаження Val...")
    val_ds = CustomLPRDataset(VAL_DIR, IMG_SIZE, CHARS_DICT)

    if len(train_ds) == 0:
        print("Помилка: Train датасет порожній.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    lprnet = LPRNet(lpr_max_len=LPR_MAX_LEN,
                    phase=True,
                    class_num=len(CHARS)+1,
                    dropout_rate=0.5).to(device)

    criterion = nn.CTCLoss(blank=len(CHARS), reduction='mean')
    optimizer = torch.optim.Adam(lprnet.parameters(), lr=LEARNING_RATE)

    print(f"--- Починаємо навчання на {EPOCHS} епох ---")

    for epoch in range(EPOCHS):
        lprnet.train()
        epoch_loss = 0
        start_time = time.time()

        for images, labels, lengths in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = lprnet(images)
            log_probs = logits.permute(2, 0, 1)
            log_probs = log_probs.log_softmax(2).requires_grad_()

            T = log_probs.shape[0]
            input_lengths, target_lengths = sparse_tuple_for_ctc(T, lengths)

            try:
                loss = criterion(log_probs, labels, input_lengths=input_lengths, target_lengths=target_lengths)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            except Exception as e:
                print(f"Помилка батчу: {e}")
                continue

        avg_loss = epoch_loss / len(train_loader)

        lprnet.eval()
        val_loss = 0
        with torch.no_grad():
            for v_imgs, v_lbls, v_lens in val_loader:
                v_imgs = v_imgs.to(device); v_lbls = v_lbls.to(device)
                v_logits = lprnet(v_imgs)
                v_probs = v_logits.permute(2, 0, 1).log_softmax(2)
                v_inp_len, v_tgt_len = sparse_tuple_for_ctc(v_probs.shape[0], v_lens)
                loss = criterion(v_probs, v_lbls, input_lengths=v_inp_len, target_lengths=v_tgt_len)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {time.time()-start_time:.1f}s")

        if (epoch + 1) % 5 == 0:
            torch.save(lprnet.state_dict(), f'{SAVE_DIR}/lprnet_epoch_{epoch+1}.pth')

    torch.save(lprnet.state_dict(), f'{SAVE_DIR}/lprnet_best.pth')
    print(f"Навчання завершено! Фінальні ваги: {SAVE_DIR}/lprnet_best.pth")

if __name__ == '__main__':
    run_training()

Модель LPRNet успішно імпортовано.
Використовуємо пристрій: cuda
Завантаження Train...
Знайдено файлів у lpr_datasets_v2/lpr_crops/train: 12521
Завантаження Val...
Знайдено файлів у lpr_datasets_v2/lpr_crops/val: 1586
--- Починаємо навчання на 10 епох ---
Epoch 1/10 | Train Loss: 1.6569 | Val Loss: 0.7445 | Time: 150.3s
Epoch 2/10 | Train Loss: 0.3161 | Val Loss: 0.2617 | Time: 111.1s
Epoch 3/10 | Train Loss: 0.1573 | Val Loss: 0.1495 | Time: 110.7s
Epoch 4/10 | Train Loss: 0.1152 | Val Loss: 0.1330 | Time: 113.4s
Epoch 5/10 | Train Loss: 0.0959 | Val Loss: 0.1592 | Time: 117.3s
Epoch 6/10 | Train Loss: 0.0796 | Val Loss: 0.0624 | Time: 110.6s
Epoch 7/10 | Train Loss: 0.0620 | Val Loss: 0.0634 | Time: 112.8s
Epoch 8/10 | Train Loss: 0.0527 | Val Loss: 0.0704 | Time: 102.1s
Epoch 9/10 | Train Loss: 0.0500 | Val Loss: 0.0456 | Time: 103.9s
Epoch 10/10 | Train Loss: 0.0483 | Val Loss: 0.0752 | Time: 142.0s
Навчання завершено! Фінальні ваги: weights_v2/lprnet_best.pth
